# MLB Betting System — Full Diagnostic Notebook

Run any section independently to verify that source is working. Each section is self-contained.

| # | Section | What it tests |
|---|---------|---------------|
| 0 | Environment | imports, API keys, repo path |
| 1 | Data lake status | DuckDB views + row counts |
| 2 | The Odds API | live sportsbook odds |
| 3 | Kalshi | prediction market odds (full diagnostic) |
| 4 | Polymarket | prediction market odds |
| 5 | Rotowire lineups | today's lineup scrape |
| 6 | MLB schedule | today's games via MLB-StatsAPI |
| 7 | pybaseball — FanGraphs | leaderboard data (season totals) |
| 8 | pybaseball — BR game logs | daily batting/pitching logs |
| 9 | Reference data | player ID map, linear weights parquets |
| 10 | Player fingerprinting | KNN comp lookup for one player |
| 11 | Projections | ProjectionEngine wOBA for a few players |
| 12 | Monte Carlo | simulate one game |
| 13 | Edge detection | run edge detection on latest lineups + odds |
| 14 | Summary | pass/fail table for all sources |

---
## 0 — Environment Setup

In [1]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import json
import time
import traceback
import requests
import duckdb
import statsapi
from datetime import datetime, timezone, date
from collections import Counter

from config.settings import (
    THE_ODDS_API_KEY, BASE_DIR,
    BRONZE_ODDS_DIR, BRONZE_LINEUPS_DIR, SILVER_LINEUPS_DIR,
    SCHEDULES_DIR, GAME_BY_GAME_DIR, FANGRAPHS_DIR,
    REFERENCE_DIR, PLAYER_ID_MAP_PATH, LINEAR_WEIGHTS_PATH,
)

TODAY = date.today()
NOW_UTC = datetime.now(timezone.utc)

# Track pass/fail for summary section
_results = {}

print(f"Repo root : {REPO_ROOT}")
print(f"Today     : {TODAY}")
print(f"UTC now   : {NOW_UTC.isoformat()}")
print(f"Odds API key : {'✅ loaded (' + THE_ODDS_API_KEY[:6] + '...)' if THE_ODDS_API_KEY else '❌ MISSING — check .env'}")

Repo root : /Users/patrickmaloney/Documents/mlb-betting
Today     : 2026-03-27
UTC now   : 2026-03-27T16:54:22.193733+00:00
Odds API key : ✅ loaded (ac12a8...)


---
## 1 — Data Lake Status (DuckDB views + row counts)

In [2]:
from src.database.db_manager import _VIEWS, register_views, BASE_DIR as DB_BASE

con = duckdb.connect()  # in-memory to avoid lock conflicts with the bot
register_views(con)

print(f"{'View':<28} {'Rows':>10}  {'Files':>5}  Pattern")
print("-" * 80)
for view_name, rel_pattern in _VIEWS:
    files = sorted(DB_BASE.glob(rel_pattern))
    if not files:
        print(f"  {view_name:<26} {'(no files)':>10}  {'0':>5}  {rel_pattern}")
        continue
    try:
        n = con.execute(f"SELECT COUNT(*) FROM {view_name}").fetchone()[0]
        print(f"  {view_name:<26} {n:>10,}  {len(files):>5}  {rel_pattern}")
    except Exception as e:
        print(f"  {view_name:<26} {'(error)':>10}  {len(files):>5}  {e}")
con.close()

View                               Rows  Files  Pattern
--------------------------------------------------------------------------------
  v_odds_the_odds_api               788      1  data/bronze/odds/the_odds_api_*.parquet
  v_odds_kalshi                     380      1  data/bronze/odds/kalshi_*.parquet
  v_odds_polymarket                 400      1  data/bronze/odds/polymarket_*.parquet
  v_bronze_lineups                1,362      1  data/bronze/lineups/lineups_*.parquet
  v_silver_lineups                  138      1  data/silver/lineups/lineups_*.parquet
  v_schedules                    73,492     26  data/schedules/*.parquet
  v_game_logs_batting           425,010      9  data/player_logs/game_by_game/batting_*.parquet
  v_game_logs_pitching          175,287      9  data/player_logs/game_by_game/pitching_*.parquet
  v_fangraphs_batting            49,482     46  data/player_logs/fangraphs_leaderboards/batting_*.parquet
  v_fangraphs_pitching           27,753     46  data/player_log

---
## 2 — The Odds API

In [5]:
try:
    resp = requests.get(
        "https://api.the-odds-api.com/v4/sports/baseball_mlb/odds",
        params={"apiKey": THE_ODDS_API_KEY, "regions": "us",
                "markets": "h2h,spreads,totals", "oddsFormat": "american"},
        timeout=15,
    )
    print(f"HTTP {resp.status_code}  |  credits remaining: {resp.headers.get('x-requests-remaining', 'n/a')}")
    games_odds = resp.json() if resp.ok else []
    print(f"Games returned: {len(games_odds)}")

    if games_odds:
        print(f"\n{'Away':<24} {'Home':<24} {'Commence (UTC)':<22} {'Books':>5}")
        print("-" * 80)
        for g in games_odds:
            print(f"{g.get('away_team',''):<24}{g.get('home_team',''):<24}"
                  f"{g.get('commence_time',''):<22}{len(g.get('bookmakers',[])):>5}")
        # Show one raw game (trimmed to 1 book)
        g0 = games_odds[0]
        g0_trim = {**g0, "bookmakers": g0.get("bookmakers", [])[:1]}
        print("\nFirst game raw (1 book):")
        print(json.dumps(g0_trim, indent=2))
        _results["The Odds API"] = f"✅ {len(games_odds)} games"
    else:
        print(f"Body: {resp.text[:400]}")
        _results["The Odds API"] = f"❌ 0 games (HTTP {resp.status_code})"
except Exception as e:
    print(f"ERROR: {e}")
    _results["The Odds API"] = f"❌ {e}"

HTTP 200  |  credits remaining: 127
Games returned: 8

Away                     Home                     Commence (UTC)         Books
--------------------------------------------------------------------------------
New York Yankees        San Francisco Giants    2026-03-27T20:36:00Z      9
Athletics               Toronto Blue Jays       2026-03-27T23:07:00Z      9
Colorado Rockies        Miami Marlins           2026-03-27T23:10:00Z      9
Kansas City Royals      Atlanta Braves          2026-03-27T23:15:00Z      9
Los Angeles Angels      Houston Astros          2026-03-28T00:16:00Z      9
Detroit Tigers          San Diego Padres        2026-03-28T01:41:00Z      9
Cleveland Guardians     Seattle Mariners        2026-03-28T01:46:00Z      9
Arizona Diamondbacks    Los Angeles Dodgers     2026-03-28T02:11:00Z      9

First game raw (1 book):
{
  "id": "487285d7a2811bab184f43dff7b4f563",
  "sport_key": "baseball_mlb",
  "sport_title": "MLB",
  "commence_time": "2026-03-27T20:36:00Z",
  "home

---
## 3 — Kalshi (full diagnostic)

In [7]:
KALSHI_HOST = "https://api.elections.kalshi.com/trade-api/v2"
ks = requests.Session()
ks.headers.update({"Accept": "application/json"})

def k_get(path, params=None):
    r = ks.get(f"{KALSHI_HOST}{path}", params=params or {}, timeout=30)
    return r

# --- 3A: Discover MLB series tickers via /series ---
r_series = k_get("/series", {"limit": 200})
print(f"GET /series → HTTP {r_series.status_code}")
all_series = r_series.json().get("series", []) if r_series.ok else []
mlb_series = [s for s in all_series
              if any(k in (s.get("title","") + s.get("ticker","")).upper()
                     for k in ["MLB", "BASEBALL"])]
                    #  for k in ["MLB", "BASEBALL", "KXML"])]
print(f"MLB-related series found: {len(mlb_series)}")
for s in mlb_series:
    print(f"  {s.get('ticker',''):<24} {s.get('title','')}")

GET /series → HTTP 200
MLB-related series found: 131
  KXMLBWS                  MLB World Series
  KXMLBWINS-NYM            Pro baseball wins New York M
  KXMLBWINS-PHI            Pro baseball wins Philadelphia
  KXMLBNLCY                Pro Baseball National League Cy Young
  KXMLBNLROTY              Pro Baseball National League Rookie of the Year
  KXMLBSTAT                Pro Baseball Season Stat
  KXMLB                    World Series
  KXMLBWINS-BAL            Pro baseball wins Baltimore
  KXMLBWINS-ATL            Pro baseball wins Atlanta
  KXMLBALROTY              Pro Baseball American League Rookie of the Year
  KXLEADERMLBWINS          MLB Wins Leader
  KXMLBHRDERBY             Pro Baseball Homerun Derby
  KXMLBHRR                 Pro Baseball Hits Runs RBIs
  KXMLBWINS-DET            Pro baseball wins Detroit
  KXMLBNLEAST              National League East Winner
  KXLEADERMLBERA           MLB ERA Leader
  KXWSNL                   MLB National League champion
  KXLEADERMLBRUN

In [8]:
# --- 3B: Probe each series for open markets ---
print(f"{'Series ticker':<24} {'open':>6} {'no filter':>10}")
print("-" * 45)
for s in mlb_series:
    ticker = s.get("ticker", "")
    r_open = k_get("/markets", {"limit": 200, "series_ticker": ticker, "status": "open"})
    n_open = len(r_open.json().get("markets", [])) if r_open.ok else "err"
    r_all  = k_get("/markets", {"limit": 200, "series_ticker": ticker})
    n_all  = len(r_all.json().get("markets", [])) if r_all.ok else "err"
    print(f"  {ticker:<22} {str(n_open):>6} {str(n_all):>10}")
    time.sleep(0.2)

Series ticker              open  no filter
---------------------------------------------
  KXMLBWS                     0          0
  KXMLBWINS-NYM               7         12
  KXMLBWINS-PHI               7         12
  KXMLBNLCY                  44        err
  KXMLBNLROTY                30        err
  KXMLBSTAT                   8        err
  KXMLB                      30        err
  KXMLBWINS-BAL               7        err
  KXMLBWINS-ATL               7        err
  KXMLBALROTY                27        err
  KXLEADERMLBWINS            50         50
  KXMLBHRDERBY                0        err
  KXMLBHRR                  200        err
  KXMLBWINS-DET               7        err
  KXMLBNLEAST                 5        err
  KXLEADERMLBERA             50        err
  KXWSNL                      0        err
  KXLEADERMLBRUNS            62        err
  KXNCAABBTOTAL               0        err
  KXMLBWORSTRECORD           30         30
  KXMLBWINS-BOS             err         12
  KXMLBW

In [9]:
# --- 3C: Full paginated fetch of KXMLBSTGAME (no status filter) ---
print("Paginating ALL KXMLBSTGAME markets...")
kalshi_all = []
cursor = None
page = 0
while True:
    params = {"limit": 1000, "series_ticker": "KXMLBSTGAME"}
    if cursor:
        params["cursor"] = cursor
    r_p = ks.get(f"{KALSHI_HOST}/markets", params=params, timeout=30)
    r_p.raise_for_status()
    d_p = r_p.json()
    batch = d_p.get("markets", [])
    kalshi_all.extend(batch)
    cursor = d_p.get("cursor")
    page += 1
    print(f"  page {page}: {len(batch)} markets  (total: {len(kalshi_all)})")
    if not cursor or not batch:
        break
    time.sleep(0.25)

status_counts = Counter(m.get("status") for m in kalshi_all)
print(f"\nTotal: {len(kalshi_all)} | Status breakdown: {dict(status_counts.most_common())}")

Paginating ALL KXMLBSTGAME markets...
  page 1: 907 markets  (total: 907)

Total: 907 | Status breakdown: {'finalized': 907}


In [24]:
# --- 3D: Find today's markets by date in ticker ---
month_map = {1:'JAN',2:'FEB',3:'MAR',4:'APR',5:'MAY',6:'JUN',
             7:'JUL',8:'AUG',9:'SEP',10:'OCT',11:'NOV',12:'DEC'}
date_pfx = f"{str(TODAY.year)[2:]}{month_map[TODAY.month]}{TODAY.day:02d}"
print(f"Looking for tickers containing: {date_pfx}")
todays_k = [m for m in kalshi_all if date_pfx in m.get("ticker", "")]
print(f"Today's markets: {len(todays_k)}")
for m in todays_k:
    print(f"  {m.get('ticker'):<45} status={m.get('status'):<12} "
          f"yes_ask={m.get('yes_ask')}  yes_bid={m.get('yes_bid')}")

open_k = [m for m in kalshi_all if m.get("status") in ("open", "active", "unopened")]
if open_k:
    print("\nFirst tradeable market:")
    print(json.dumps(open_k[0], indent=2, default=str))

_results["Kalshi"] = (
    f"✅ {len(todays_k)} today's markets (total {len(kalshi_all)}, "
    + ", ".join(f"{s}={c}" for s, c in status_counts.most_common()) + ")"
    if kalshi_all else "❌ 0 markets fetched"
)

Looking for tickers containing: 26MAR25
Today's markets: 0


In [34]:
# What markets are actually open RIGHT NOW (no series filter)?
r_open_all = ks.get(f"{KALSHI_HOST}/markets",
                    params={"limit": 500, "status": "open"}, timeout=30)                                                                     
open_all = r_open_all.json().get("markets", [])                                                                                              
print(f"Open markets (no filter): {len(open_all)}")                                                                                          
                                                                                                                                            
# Show unique series tickers among open markets           
open_series = Counter(m.get("series_ticker") or m.get("event_ticker","").split("-")[0]                                                       
                    for m in open_all)                                                                                                     
print("Series breakdown:", dict(open_series.most_common()))
                                                                                                                                            
# Show any with MLB-related tickers                       
mlb_open = [m for m in open_all                                                                                                              
            if any(k in m.get("ticker","").upper() for k in ["MLB","KXML","BASEBALL"])]                                                      
print(f"\nMLB-looking open markets: {len(mlb_open)}")                                                                                        
for m in mlb_open[:10]:                                                                                                                      
    print(f"  {m.get('ticker')}") 

Open markets (no filter): 100
Series breakdown: {'KXMVESPORTSMULTIGAMEEXTENDED': 75, 'KXMVECROSSCATEGORY': 25}

MLB-looking open markets: 0


---
## 4 — Polymarket

In [25]:
try:
    import re as _re
    pm_events = []
    offset = 0
    while True:
        r_pm = requests.get(
            "https://gamma-api.polymarket.com/events",
            params={"tag_slug": "mlb", "active": "true", "closed": "false",
                    "limit": 100, "offset": offset},
            timeout=30,
        )
        r_pm.raise_for_status()
        batch = r_pm.json()
        if not batch:
            break
        pm_events.extend(batch)
        if len(batch) < 100:
            break
        offset += 100

    game_ev  = [e for e in pm_events if " vs. " in e.get("title", "")]
    wt_ev    = [e for e in pm_events if "Regular Season" in e.get("title", "")]
    game_mkts = sum(len(e.get("markets", [])) for e in game_ev)
    wt_mkts   = sum(len(e.get("markets", [])) for e in wt_ev)

    print(f"Events fetched   : {len(pm_events)}")
    print(f"Game events      : {len(game_ev)} ({game_mkts} markets)")
    print(f"Win-total events : {len(wt_ev)} ({wt_mkts} markets)")

    if game_ev:
        print(f"\n{'Date':<12} {'Event title':<50} {'Mkts':>4}")
        print("-" * 70)
        for e in game_ev[:15]:
            d = str(e.get("eventDate") or e.get("endDate", ""))[:10]
            print(f"{d:<12} {e.get('title','')[:48]:<50} {len(e.get('markets',[])):>4}")

    _results["Polymarket"] = f"✅ {len(game_ev)} game events, {game_mkts} markets"
except Exception as e:
    print(f"ERROR: {e}")
    _results["Polymarket"] = f"❌ {e}"

Events fetched   : 111
Game events      : 85 (185 markets)
Win-total events : 1 (30 markets)

Date         Event title                                        Mkts
----------------------------------------------------------------------
2026-03-07   Tampa Bay Rays vs. Boston Red Sox                     4
2026-03-08   St. Louis Cardinals vs. Miami Marlins                 3
2026-03-27   New York Yankees vs. San Francisco Giants             5
2026-03-27   Athletics vs. Toronto Blue Jays                       5
2026-03-27   Colorado Rockies vs. Miami Marlins                    5
2026-03-27   Kansas City Royals vs. Atlanta Braves                 5
2026-03-27   Los Angeles Angels vs. Houston Astros                 2
2026-03-27   Detroit Tigers vs. San Diego Padres                   2
2026-03-27   Cleveland Guardians vs. Seattle Mariners              2
2026-03-27   Arizona Diamondbacks vs. Los Angeles Dodgers          2
2026-03-28   Tampa Bay Rays vs. St. Louis Cardinals                2
2026-03

---
## 5 — Rotowire Lineups

In [34]:
try:
    from src.data_ingestion.lineups import fetch_lineups, save_lineups
    df_lineups = fetch_lineups(live=True)
    if df_lineups is not None and not df_lineups.empty:
        print(f"\nRows returned  : {len(df_lineups):,}")
        if "game_date" in df_lineups.columns and "away_team" in df_lineups.columns:
            games = df_lineups.groupby(["game_date", "away_team", "home_team"]).size()
            print(f"Games found    : {len(games)}")
            print(games.reset_index(drop=False).to_string(index=False))
        print("\nSample (first 3 rows):")
        print(df_lineups.head(3).to_string())

        # Save to bronze + silver so downstream cells (Monte Carlo, edge detection) have data
        save_lineups(df_lineups)
        print("\n✅ Lineups saved to bronze + silver parquet")
        _results["Lineups (Rotowire)"] = f"✅ {len(df_lineups):,} rows, {len(games)} games — saved"
    else:
        print("No data returned.")
        _results["Lineups (Rotowire)"] = "❌ empty result"
except Exception as e:
    traceback.print_exc()
    _results["Lineups (Rotowire)"] = f"❌ {e}"

✅ Fetched live Rotowire daily lineups
✅ Parsed 8 games
✅ Loading existing player_id_map from /Users/patrickmaloney/Documents/mlb-betting/data/reference/player_id_map.parquet
No identically matched names found! Returning the 5 most similar names.
No identically matched names found! Returning the 5 most similar names.
No identically matched names found! Returning the 5 most similar names.
No identically matched names found! Returning the 5 most similar names.
No identically matched names found! Returning the 5 most similar names.
No identically matched names found! Returning the 5 most similar names.
No identically matched names found! Returning the 5 most similar names.
No identically matched names found! Returning the 5 most similar names.
No identically matched names found! Returning the 5 most similar names.
No identically matched names found! Returning the 5 most similar names.
No identically matched names found! Returning the 5 most similar names.
No identically matched names found

---
## 6 — MLB Schedule (MLB-StatsAPI)

In [27]:
try:
    today_str = TODAY.strftime("%Y-%m-%d")
    schedule = statsapi.schedule(start_date=today_str, end_date=today_str)
    print(f"Today ({today_str}) — {len(schedule)} games via MLB-StatsAPI")

    if schedule:
        print(f"\n{'Away':<28} {'Home':<28} {'Status':<14} {'Type'}")
        print("-" * 80)
        for g in schedule:
            print(f"{g.get('away_name',''):<28}{g.get('home_name',''):<28}"
                  f"{g.get('status',''):<14}{g.get('game_type','')}")

    # Also check existing schedule parquets
    sched_files = sorted(SCHEDULES_DIR.glob("*.parquet"))
    print(f"\nSchedule parquets on disk: {len(sched_files)}")
    for f in sched_files:
        print(f"  {f.name}")

    _results["MLB-StatsAPI"] = f"✅ {len(schedule)} games today"
except Exception as e:
    traceback.print_exc()
    _results["MLB-StatsAPI"] = f"❌ {e}"

Today (2026-03-27) — 8 games via MLB-StatsAPI

Away                         Home                         Status         Type
--------------------------------------------------------------------------------
New York Yankees            San Francisco Giants        Pre-Game      R
Athletics                   Toronto Blue Jays           Scheduled     R
Colorado Rockies            Miami Marlins               Scheduled     R
Kansas City Royals          Atlanta Braves              Pre-Game      R
Los Angeles Angels          Houston Astros              Scheduled     R
Detroit Tigers              San Diego Padres            Scheduled     R
Cleveland Guardians         Seattle Mariners            Scheduled     R
Arizona Diamondbacks        Los Angeles Dodgers         Scheduled     R

Schedule parquets on disk: 26
  games_2000.parquet
  games_2001.parquet
  games_2002.parquet
  games_2003.parquet
  games_2004.parquet
  games_2005.parquet
  games_2006.parquet
  games_2007.parquet
  games_2008.parque

---
## 7 — pybaseball: FanGraphs Leaderboards

In [38]:
# Check local FanGraphs parquets first (what we have on disk)
fg_batting  = sorted(FANGRAPHS_DIR.glob("batting_*.parquet"))
fg_pitching = sorted(FANGRAPHS_DIR.glob("pitching_*.parquet"))
print(f"FanGraphs batting  parquets : {len(fg_batting)}")
print(f"FanGraphs pitching parquets : {len(fg_pitching)}")
if fg_batting:
    years = [f.stem.split("_")[-1] for f in fg_batting]
    print(f"Batting years on disk : {years}")

# Use 2025 for the connectivity test — FanGraphs leaderboards return malformed HTML
# early in a new season (column headers exist but no data rows), which breaks pandas.
print("\nTesting pybaseball FanGraphs API connectivity (using 2025)...")
try:
    import pybaseball as pb
    pb.cache.enable()
    sample = pb.batting_stats(2025, qual=200)
    if sample is not None and not sample.empty:
        print(f"  2025 FanGraphs batting leaders: {len(sample)} players (qual ≥ 200 PA)")
        cols = [c for c in ["Name", "Team", "PA", "wRC+", "wOBA", "ISO", "K%", "BB%"] if c in sample.columns]
        print(sample[cols].head(5).to_string(index=False))
        _results["pybaseball FanGraphs"] = f"✅ {len(sample)} players (2025)"
    else:
        print("  No data returned.")
        _results["pybaseball FanGraphs"] = "❌ no data returned"
except Exception as e:
    traceback.print_exc()
    _results["pybaseball FanGraphs"] = f"❌ {e}"

FanGraphs batting  parquets : 46
FanGraphs pitching parquets : 46
Batting years on disk : ['1980', '1981', '1982', '1983', '1984', '1985', '1986', '1987', '1988', '1989', '1990', '1991', '1992', '1993', '1994', '1995', '1996', '1997', '1998', '1999', '2000', '2001', '2002', '2003', '2004', '2005', '2006', '2007', '2008', '2009', '2010', '2011', '2012', '2013', '2014', '2015', '2016', '2017', '2018', '2019', '2020', '2021', '2022', '2023', '2024', '2025']

Testing pybaseball FanGraphs API connectivity (using 2025)...
  2025 FanGraphs batting leaders: 348 players (qual ≥ 200 PA)
           Name Team  PA  wRC+  wOBA   ISO    K%   BB%
    Aaron Judge  NYY 679   204 0.463 0.357 0.236 0.183
    Cal Raleigh  SEA 705   161 0.392 0.342 0.267 0.138
 Bobby Witt Jr.  KCR 687   130 0.360 0.205 0.182 0.071
  Shohei Ohtani  LAD 727   172 0.418 0.340 0.257 0.150
Geraldo Perdomo  ARI 720   138 0.370 0.173 0.115 0.131


---
## 8 — pybaseball: Baseball Reference Game Logs

In [20]:
# Check what game-log parquets already exist
gbg_batting  = sorted(GAME_BY_GAME_DIR.glob("batting_*.parquet"))
gbg_pitching = sorted(GAME_BY_GAME_DIR.glob("pitching_*.parquet"))
print(f"Game-by-game batting  parquets : {len(gbg_batting)}")
print(f"Game-by-game pitching parquets : {len(gbg_pitching)}")

if gbg_batting:
    years = [f.stem.split("_")[-1] for f in gbg_batting]
    print(f"Years on disk : {years}")

    # Verify each file is readable and get row counts
    con_tmp = duckdb.connect()
    print(f"\n{'File':<40} {'Rows':>8}")
    print("-" * 50)
    total_batting = 0
    for f in gbg_batting:
        try:
            n = con_tmp.execute(f"SELECT COUNT(*) FROM read_parquet('{f}')").fetchone()[0]
            print(f"  {f.name:<38} {n:>8,}")
            total_batting += n
        except Exception as e:
            print(f"  {f.name:<38} ERROR: {e}")
    con_tmp.close()
    print(f"  {'TOTAL':<38} {total_batting:>8,}")
    _results["BR Game Logs"] = f"✅ {len(gbg_batting)} parquets on disk, {total_batting:,} rows (2017–2025)"
else:
    print("No game-log parquets found.")
    _results["BR Game Logs"] = "❌ no parquets on disk"

# NOTE: pb.batting_stats_range() is broken — BR changed their HTML structure and
# pybaseball's scraper crashes with IndexError regardless of date. The local
# parquets are the authoritative data source; live BR fetching is not used in
# the daily pipeline (player_logs_fetcher.py is run manually for historical data).

Game-by-game batting  parquets : 9
Game-by-game pitching parquets : 9
Years on disk : ['2017', '2018', '2019', '2020', '2021', '2022', '2023', '2024', '2025']

File                                         Rows
--------------------------------------------------
  batting_game_logs_2017.parquet           51,483
  batting_game_logs_2018.parquet           52,505
  batting_game_logs_2019.parquet           52,477
  batting_game_logs_2020.parquet           17,886
  batting_game_logs_2021.parquet           51,850
  batting_game_logs_2022.parquet           49,036
  batting_game_logs_2023.parquet           49,377
  batting_game_logs_2024.parquet           50,130
  batting_game_logs_2025.parquet           50,266
  TOTAL                                   425,010


---
## 9 — Reference Data (Player ID Map + Linear Weights)

In [22]:
con_ref = duckdb.connect()

# --- Player ID Map ---
print("=== Player ID Map ===")
if PLAYER_ID_MAP_PATH.exists():
    n_id = con_ref.execute(f"SELECT COUNT(*) FROM read_parquet('{PLAYER_ID_MAP_PATH}')").fetchone()[0]
    cols_id = con_ref.execute(f"DESCRIBE SELECT * FROM read_parquet('{PLAYER_ID_MAP_PATH}')").fetchall()
    n_with_fg = con_ref.execute(
        f"SELECT COUNT(*) FROM read_parquet('{PLAYER_ID_MAP_PATH}') WHERE IDFANGRAPHS IS NOT NULL"
    ).fetchone()[0]
    n_with_bref = con_ref.execute(
        f"SELECT COUNT(*) FROM read_parquet('{PLAYER_ID_MAP_PATH}') WHERE BREFID IS NOT NULL"
    ).fetchone()[0]
    print(f"  File      : {PLAYER_ID_MAP_PATH.name}")
    print(f"  Total rows: {n_id:,}")
    print(f"  w/ IDFANGRAPHS: {n_with_fg:,}")
    print(f"  w/ BREFID     : {n_with_bref:,}")
    print(f"  Columns : {[c[0] for c in cols_id]}")
    sample_id = con_ref.execute(
        f"SELECT PLAYERNAME, BREFID, IDFANGRAPHS, TEAM FROM read_parquet('{PLAYER_ID_MAP_PATH}') LIMIT 5"
    ).fetchall()
    print("  Sample:")
    for row in sample_id:
        print(f"    {row}")
    _results["Player ID Map"] = f"✅ {n_id:,} rows ({n_with_fg:,} w/ FG ID)"
else:
    print(f"  ❌ NOT FOUND: {PLAYER_ID_MAP_PATH}")
    _results["Player ID Map"] = "❌ file missing"

print()

# --- Linear Weights ---
print("=== Linear Weights ===")
if LINEAR_WEIGHTS_PATH.exists():
    n_lw = con_ref.execute(f"SELECT COUNT(*) FROM read_parquet('{LINEAR_WEIGHTS_PATH}')").fetchone()[0]
    # Discover actual column names first
    lw_cols = [c[0] for c in con_ref.execute(f"DESCRIBE SELECT * FROM read_parquet('{LINEAR_WEIGHTS_PATH}')").fetchall()]
    season_col = next((c for c in lw_cols if c.lower() == "season"), None)
    print(f"  File       : {LINEAR_WEIGHTS_PATH.name}")
    print(f"  Total rows : {n_lw:,}")
    print(f"  Columns    : {lw_cols}")
    if season_col:
        yr_range = con_ref.execute(
            f'SELECT MIN("{season_col}"), MAX("{season_col}") FROM read_parquet(\'{LINEAR_WEIGHTS_PATH}\')'
        ).fetchone()
        print(f"  Year range : {yr_range[0]}–{yr_range[1]}")
        _results["Linear Weights"] = f"✅ {n_lw:,} rows ({yr_range[0]}–{yr_range[1]})"
    else:
        _results["Linear Weights"] = f"✅ {n_lw:,} rows (no Season column found)"
else:
    print(f"  ❌ NOT FOUND: {LINEAR_WEIGHTS_PATH}")
    _results["Linear Weights"] = "❌ file missing"

con_ref.close()

=== Player ID Map ===
  File      : player_id_map.parquet
  Total rows: 3,045
  w/ IDFANGRAPHS: 3,041
  w/ BREFID     : 3,044
  Columns : ['PLAYERNAME', 'BIRTHDATE', 'BATS', 'THROWS', 'POS', 'TEAM', 'BREFID', 'MLBID', 'ROTOWIRENAME', 'ROTOWIREID', 'IDFANGRAPHS', 'FANGRAPHSNAME', 'name_norm', 'playername_norm', 'bbref_id', 'mlbam_id', 'rotowire_id']
  Sample:
    ('David Aardsma', 'aardsda01', '1902', None)
    ('Fernando Abad', 'abadfe01', '4994', None)
    ('Cory Abbott', 'abbotco01', 'sa3005305', 'SF')
    ('Mick Abel', 'abelmi01', 'sa3014534', 'PHI')
    ('CJ Abrams', 'abramcj01', '25768', 'WAS')

=== Linear Weights ===
  File       : linear_weights.parquet
  Total rows : 155
  Columns    : ['"Season"', '"wOBA"', '"wOBAScale"', '"wBB"', '"wHBP"', '"w1B"', '"w2B"', '"w3B"', '"wHR"', '"runSB"', '"runCS"', '"R/PA"', '"R/W"', '"cFIP"', 'date']


---
## 10 — Player Fingerprinting (KNN Comps)

In [ ]:
try:
    from src.features.player_fingerprinting import PlayerComps
    pc = PlayerComps()
    pc.fit(is_batter=True)
    n_samples = pc._hist_features.shape[0]
    print(f"PlayerComps fitted on {n_samples:,} batter-seasons")

    # Test with Mookie Betts' approximate 2025 profile
    test_player = {
        "name": "Mookie Betts",
        "age": 32,
        "service_time": 11,
        "pa": 550,
        "wrc_plus": 145,
        "iso": 0.220,
        "k_pct": 0.15,
        "bb_pct": 0.10,
        "babip": 0.295,
        "hardhit_pct": 0.42,
        "barrel_pct": 0.12,
        "spd": 5.5,
    }
    comps, proj = pc.get_comps(test_player)   # param is 'n', not 'k'; default is 15
    print(f"\nMookie Betts — top {len(comps)} comps:")
    for i, c in enumerate(comps[:5], 1):
        name = c.get("player_name") or c.get("name", "?")
        print(f"  {i}. {name:25} {c.get('season','')}  wRC+={c.get('wrc_plus','?')}")
    print(f"\nProjected stats: {proj}")
    _results["Player Fingerprinting"] = f"✅ fitted on {n_samples:,} seasons"
except Exception as e:
    traceback.print_exc()
    _results["Player Fingerprinting"] = f"❌ {e}"

---
## 11 — Projection Engine

In [35]:
try:
    from src.models.projections import ProjectionEngine
    print("Initializing ProjectionEngine (may take ~30s first time)...")
    pe = ProjectionEngine()
    print(f"ProjectionEngine ready — {len(pe._player_woba):,} L0 players, {len(pe._knn_woba):,} L1 (KNN) players")

    # Spot-check a few known players
    test_ids = [
        ("bettmo01",  "Mookie Betts"),
        ("troutmi01", "Mike Trout"),
        ("ohtansh01", "Shohei Ohtani"),
        ("judgaar01", "Aaron Judge"),
    ]
    print(f"\n{'Player':<20} {'bbref_id':<12} {'wOBA':>6}  {'Layer'}")
    print("-" * 50)
    for bbref, name in test_ids:
        woba = pe.get_player_woba(bbref)
        layer = "L1" if bbref in pe._knn_woba else ("L0" if pe._bbref_to_mlbid.get(bbref) in pe._player_woba else "avg")
        print(f"{name:<20} {bbref:<12} {woba:.3f}  {layer}")

    _results["Projection Engine"] = f"✅ {len(pe._player_woba):,} L0, {len(pe._knn_woba):,} L1 players"
except Exception as e:
    traceback.print_exc()
    _results["Projection Engine"] = f"❌ {e}"

Initializing ProjectionEngine (may take ~30s first time)...
📊 Building player projections...
   155 seasons of linear weights loaded
   2,309 players with career wOBA (L0)
   3,041 BBRef → mlbID mappings
   3,040 BBRef → IDfg mappings
   1,548 players with FG season stats for KNN
   📂 Loaded 5873 batter player-seasons from cache
✅ KNN fitted for batters (5873 samples, 11 features)
✅ Projections ready: 2,309 players (L0), 1,253 KNN projections (L1), league avg wOBA = 0.316
ProjectionEngine ready — 2,309 L0 players, 1,253 L1 (KNN) players

Player               bbref_id       wOBA  Layer
--------------------------------------------------
Mookie Betts         bettmo01     0.316  avg
Mike Trout           troutmi01    0.366  L1
Shohei Ohtani        ohtansh01    0.464  L1
Aaron Judge          judgaar01    0.316  avg


---
## 12 — Monte Carlo Simulation

In [ ]:
try:
    from src.models.monte_carlo import MonteCarloSimulator

    sim = MonteCarloSimulator()
    today_str = TODAY.strftime("%Y-%m-%d")

    # Try today first, then fall back to most recent date in silver
    results = sim.run_all(today_str)

    if not results:
        print(f"No results for today ({today_str}). Checking most recent silver date...")
        con_mc = duckdb.connect()
        from src.database.db_manager import register_views
        register_views(con_mc)
        most_recent = con_mc.execute("SELECT MAX(game_date) FROM v_silver_lineups").fetchone()[0]
        con_mc.close()
        if most_recent and str(most_recent) != today_str:
            print(f"Using {most_recent}")
            results = sim.run_all(str(most_recent))

    if results:
        print(f"\n{'Away':<6} {'Home':<6} {'Away RS':>7} {'Home RS':>7} {'Away W%':>7} {'Home W%':>7}")
        print("-" * 48)
        for r in results:
            print(f"{r['away_team']:<6} {r['home_team']:<6} "
                  f"{r['away_runs_proj']:>7.2f} {r['home_runs_proj']:>7.2f} "
                  f"{r['away_win_prob']:>7.1%} {r['home_win_prob']:>7.1%}")
        _results["Monte Carlo"] = f"✅ {len(results)} games simulated"
    else:
        print("No simulable games found — silver lineups may be empty. Run cell 5 first.")
        _results["Monte Carlo"] = "⚠️  no simulable games"
except Exception as e:
    traceback.print_exc()
    _results["Monte Carlo"] = f"❌ {e}"

---
## 13 — Edge Detection

In [ ]:
try:
    from src.models.edge_detection import find_edges

    today_str = TODAY.strftime("%Y-%m-%d")
    edges = find_edges(game_date=today_str)

    if edges:
        print(f"\nEdges found: {len(edges)}")
        print(f"\n{'Game':<32} {'Side':<6} {'Model':>6} {'Mkt':>6} {'Edge':>6}  Source")
        print("-" * 72)
        for e in edges[:20]:
            game = f"{e.get('away_team','?')}@{e.get('home_team','?')}"
            print(
                f"{game:<32}"
                f"{e.get('side',''):<6}"
                f"{e.get('model_prob',0):.1%}  "
                f"{e.get('market_prob',0):.1%}  "
                f"{e.get('edge',0):+.1%}  "
                f"{e.get('source','')}"
            )
        _results["Edge Detection"] = f"✅ {len(edges)} edges"
    else:
        print("No edges found. Likely causes: no sim results saved yet (run cell 12 first), "
              "no odds loaded for today, or all edges below 3% threshold.")
        _results["Edge Detection"] = "⚠️  0 edges"
except Exception as e:
    traceback.print_exc()
    _results["Edge Detection"] = f"❌ {e}"

---
## 14 — Summary

In [38]:
print(f"\n{'='*65}")
print(f"SYSTEM HEALTH SUMMARY  —  {TODAY}")
print(f"{'='*65}")
for source, status in _results.items():
    print(f"  {source:<28} {status}")
print(f"{'='*65}")


SYSTEM HEALTH SUMMARY  —  2026-03-27
  The Odds API                 ✅ 22 games
  Kalshi                       ✅ 0 today's markets (total 907, finalized=907)
  Polymarket                   ✅ 85 game events, 185 markets
  Lineups (Rotowire)           ✅ 8 rows, 8 games — saved
  MLB-StatsAPI                 ✅ 8 games today
  pybaseball FanGraphs         ✅ 348 players (2025)
  BR Game Logs                 ✅ 9 parquets on disk, 425,010 rows (2017–2025)
  Player ID Map                ✅ 3,045 rows (3,041 w/ FG ID)
  Linear Weights               ✅ 155 rows (no Season column found)
  Player Fingerprinting        ✅ fitted on 5,873 seasons
  Projection Engine            ✅ 2,309 L0, 1,253 L1 players
  Monte Carlo                  ❌ MonteCarloSimulator.simulate_game() takes 2 positional arguments but 4 were given
  Edge Detection               ❌ cannot import name 'EdgeDetector' from 'src.models.edge_detection' (/Users/patrickmaloney/Documents/mlb-betting/src/models/edge_detection.py)
